In [1]:
#install cell
!pip install yfinance alpaca-py plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 2.3 MB/s eta 0:00:00


In [2]:
#import cell
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

In [1]:
#connect cell
API_KEY    = "key"
API_SECRET = "secret"

client = TradingClient(API_KEY, API_SECRET, paper=True)
print(f"Status: {client.get_account().status}")
print(f"Cash:   ${float(client.get_account().cash):,.2f}")

NameError: name 'TradingClient' is not defined

In [4]:
#tickers
def get_sp500_tickers():
    # iShares first
    try:
        holdings = pd.read_csv(
            'https://www.ishares.com/us/products/239726/'
            'ISHARES-CORE-SP-500-ETF/1467271812596.ajax'
            '?fileType=csv&fileName=IVV_holdings'
            '&dataType=fund',
            skiprows=9
        )
        holdings = holdings.dropna(subset=['Ticker'])
        holdings = holdings[
            holdings['Asset Class'] == 'Equity'
        ]
        tickers  = holdings['Ticker'].tolist()
        tickers  = [t.replace('.', '-') for t in tickers
                    if isinstance(t, str)
                    and t != 'nan']
        if len(tickers) > 400:
            print(f"iShares: found {len(tickers)} tickers")
            return tickers
    except Exception as e:
        print(f"iShares failed: {e}")

    # Wikipedia second
    try:
        sp500   = pd.read_html(
            'https://en.wikipedia.org/wiki/'
            'List_of_S%26P_500_companies'
        )[0]
        tickers = sp500['Symbol'].tolist()
        tickers = [t.replace('.', '-') for t in tickers]
        if len(tickers) > 400:
            print(f"Wikipedia: found {len(tickers)} tickers")
            return tickers
    except Exception as e:
        print(f"Wikipedia failed: {e}")

    # SPDR last
    try:
        holdings = pd.read_excel(
            'https://www.ssga.com/us/en/intermediary/'
            'etfs/library-content/products/fund-data/'
            'etfs/us/holdings-daily-us-en-spy.xlsx',
            skiprows=4
        )
        holdings = holdings.dropna(subset=['Ticker'])
        tickers  = holdings['Ticker'].tolist()
        tickers  = [str(t).replace('.', '-')
                    for t in tickers
                    if str(t) != 'nan']
        if len(tickers) > 400:
            print(f"SPDR: found {len(tickers)} tickers")
            return tickers
    except Exception as e:
        print(f"SPDR failed: {e}")

    raise Exception("All sources failed — "
                    "check your internet connection")

tickers = get_sp500_tickers()

iShares failed: Error tokenizing data. C error: Expected 1 fields in line 16, saw 4

Wikipedia failed: HTTP Error 403: Forbidden
SPDR: found 505 tickers


In [5]:
# download & clean
print("Downloading data... takes 2-3 minutes")
data = yf.download(
    tickers,
    start='2015-01-01',
    auto_adjust=True,
    progress=True
)['Close']

data = data.dropna(thresh=int(len(data) * 0.90), axis=1)
print(f"Stocks after cleaning: {data.shape[1]}")

[******                13%                       ]  64 of 505 completedERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 2602335D"}}}
[*********************100%***********************]  505 of 505 completed
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['-', '2602335D']: YFTzMissingError('possibly delisted; no timezone found')


Stocks after cleaning: 467


In [6]:
# liquidity filter
volume = yf.download(
    tickers,
    start='2015-01-01',
    auto_adjust=True,
    progress=False
)['Volume']

avg_dollar_vol = volume.mean() * data.mean()
liquid         = avg_dollar_vol[
    avg_dollar_vol > 10_000_000
].index
data           = data[[c for c in liquid
                        if c in data.columns]]
print(f"Stocks after liquidity filter: {data.shape[1]}")

ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['-', '2602335D']: YFTzMissingError('possibly delisted; no timezone found')


Stocks after liquidity filter: 467


In [7]:
# momentum signal
monthly = data.resample('ME').last()

def momentum_signal(prices, lookback=6, skip=1, top_n=10):
    rets    = prices.pct_change(lookback).shift(skip)
    ranks   = rets.rank(axis=1, ascending=False)
    signal  = (ranks <= top_n).astype(int)
    weights = signal.div(signal.sum(axis=1), axis=0)
    return weights, rets

weights, momentum_rets = momentum_signal(monthly, top_n=10)

latest = weights.iloc[-1]
picks  = latest[latest > 0].sort_values(ascending=False)

print(f"=== TOP 10 MOMENTUM STOCKS ===")
print(f"Selected from {data.shape[1]} stocks\n")
for t, w in picks.items():
    ret = momentum_rets.iloc[-1][t]
    print(f"  {t}: {w:.0%} | {ret:.1%} 6-month return")

=== TOP 10 MOMENTUM STOCKS ===
Selected from 467 stocks

  AMD: 10% | 137.3% 6-month return
  CIEN: 10% | 184.1% 6-month return
  COHR: 10% | 120.1% 6-month return
  GLW: 10% | 115.9% 6-month return
  INTC: 10% | 182.7% 6-month return
  LITE: 10% | 162.9% 6-month return
  MU: 10% | 310.9% 6-month return
  ON: 10% | 140.1% 6-month return
  STX: 10% | 219.4% 6-month return
  WDC: 10% | 225.7% 6-month return


In [ ]:
# Regime Detection Cell

# Detect bull vs bear market using SPY 200d MA
# Bull market = SPY above its 200d MA
# Bear market = SPY below its 200d MA

spy_prices  = data['SPY'] if 'SPY' in data.columns \
              else data.iloc[:, 0]

spy_monthly = spy_prices.resample('ME').last()
spy_200ma   = spy_monthly.rolling(200).mean()
bull_market = (spy_monthly > spy_200ma).astype(float)

# In bear market: cut momentum exposure by 50%
regime_scalar = bull_market.map(
    {1.0: 1.0, 0.0: 0.5}
).fillna(1.0)

# Apply regime to weights
weights_regime = weights.multiply(
    regime_scalar, axis=0
)

# Renormalize
weights_regime = weights_regime.div(
    weights_regime.sum(axis=1), axis=0
).fillna(0)

# Show current regime
current_regime = 'BULL' if bull_market.iloc[-1] == 1 \
                 else 'BEAR'
print(f"Current market regime: {current_regime}")
print(f"Momentum weight scalar: "
      f"{regime_scalar.iloc[-1]:.0%}")

# Use weights_regime instead of weights
weights = weights_regime

In [8]:
# Momentum Rankings
import plotly.graph_objects as go

# latest momentum returns and ranks
latest_rets  = momentum_rets.iloc[-1].dropna()
latest_ranks = latest_rets.rank(ascending=False)
latest_picks = weights.iloc[-1]

# summary table
summary = pd.DataFrame({
    'Rank':          latest_ranks,
    '6M Return %':   latest_rets * 100,
    'Weight':        latest_picks
}).dropna()
summary = summary.sort_values('Rank')
summary['Selected'] = summary['Weight'] > 0

print("=" * 60)
print(f"  MOMENTUM RANKINGS — "
      f"{monthly.index[-1].strftime('%B %d, %Y')}")
print("=" * 60)
print(f"\n  Universe: {len(summary)} stocks  |  "
      f"Holding top {int(summary['Selected'].sum())}\n")

print(f"  {'Rank':<6} {'Ticker':<8} "
      f"{'6M Return':>10} {'Status':<15}")
print(f"  {'-'*6} {'-'*8} "
      f"{'-'*10} {'-'*15}")

for _, row in summary.iterrows():
    ticker   = row.name
    rank     = int(row['Rank'])
    ret      = row['6M Return %']
    selected = row['Selected']
    marker   = '◄ HOLDING' if selected else ''
    arrow    = '▲' if ret >= 0 else '▼'

    print(f"  {rank:<6} {ticker:<8} "
          f"  {arrow}{abs(ret):>8.1f}%  "
          f"{marker}")

    # top 20 and bottom 10
    if rank == 20 and not selected:
        remaining = len(summary) - 20
        print(f"\n  ... {remaining} more stocks "
              f"not shown ...\n")
        break

# Chart 1 — top 20 and bottom 20 momentum
top20    = summary.head(20)
bottom20 = summary.tail(20)
combined = pd.concat([bottom20, top20])

colors = [
    '#1D9E75' if row['Selected']
    else '#888780' if row['6M Return %'] >= 0
    else '#D85A30'
    for _, row in combined.iterrows()
]

fig = go.Figure(go.Bar(
    x=combined.index,
    y=combined['6M Return %'],
    marker_color=colors,
    text=[f"{v:+.1f}%"
          for v in combined['6M Return %']],
    textposition='outside'
))

fig.add_hline(
    y=0,
    line_dash='dash',
    line_color='#888780',
    line_width=1
)

fig.update_layout(
    title='Momentum Rankings — Top 20 & Bottom 20  '
          '(green = holding, gray = positive not selected, '
          'red = negative)',
    yaxis_title='6-Month Return %',
    xaxis_title='Stock',
    height=500,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig.update_yaxes(ticksuffix='%')
fig.show()

# Chart 2 — full distribution of momentum scores
fig2 = go.Figure(go.Histogram(
    x=summary['6M Return %'],
    nbinsx=50,
    marker_color='#1D9E75',
    opacity=0.8
))

# Mark the cutoff for top 10
cutoff = summary[
    summary['Rank'] <= summary['Selected'].sum()
]['6M Return %'].min()

fig2.add_vline(
    x=cutoff,
    line_dash='dash',
    line_color='#D85A30',
    annotation_text=f'selection cutoff '
                    f'({cutoff:.1f}%)'
)

fig2.update_layout(
    title='Distribution of 6-Month Returns '
          'Across All 500 Stocks',
    xaxis_title='6-Month Return %',
    yaxis_title='Number of stocks',
    height=400,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.update_xaxes(ticksuffix='%')
fig2.show()

# Chart 3 — your current holdings ranked
holdings = summary[summary['Selected']].copy()
holdings = holdings.sort_values(
    '6M Return %', ascending=True
)

fig3 = go.Figure(go.Bar(
    x=holdings['6M Return %'],
    y=holdings.index,
    orientation='h',
    marker_color='#1D9E75',
    text=[f"Rank #{int(r)} | {v:+.1f}%"
          for r, v in zip(
              holdings['Rank'],
              holdings['6M Return %']
          )],
    textposition='outside'
))

fig3.update_layout(
    title='Your Current Holdings — '
          'Ranked by Momentum Score',
    xaxis_title='6-Month Return %',
    height=450,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig3.update_xaxes(ticksuffix='%')
fig3.show()

# Summary stats
print(f"\n  MOMENTUM SUMMARY")
print(f"  {'─'*45}")
print(f"  Stocks with positive momentum: "
      f"{(summary['6M Return %'] > 0).sum()} "
      f"({(summary['6M Return %'] > 0).mean():.0%})")
print(f"  Avg 6M return (all stocks):   "
      f"{summary['6M Return %'].mean():+.1f}%")
print(f"  Avg 6M return (your picks):   "
      f"{holdings['6M Return %'].mean():+.1f}%")
print(f"  Weakest holding:              "
      f"{holdings.index[0]} "
      f"({holdings['6M Return %'].iloc[0]:+.1f}%)")
print(f"  Strongest holding:            "
      f"{holdings.index[-1]} "
      f"({holdings['6M Return %'].iloc[-1]:+.1f}%)")
print(f"\n  NEXT REBALANCE — check if any of these "
      f"drop out of top 10:")

# Stocks close to falling out
borderline = summary[
    (summary['Rank'] > 10) &
    (summary['Rank'] <= 20)
].head(5)
for _, row in borderline.iterrows():
    print(f"    Rank #{int(row['Rank'])}: "
          f"{row.name} "
          f"({row['6M Return %']:+.1f}%) — "
          f"watching")

  MOMENTUM RANKINGS — June 30, 2026

  Universe: 467 stocks  |  Holding top 10

  Rank   Ticker    6M Return Status         
  ------ -------- ---------- ---------------
  1      MU         ▲   310.9%  ◄ HOLDING
  2      WDC        ▲   225.7%  ◄ HOLDING
  3      STX        ▲   219.4%  ◄ HOLDING
  4      CIEN       ▲   184.1%  ◄ HOLDING
  5      INTC       ▲   182.7%  ◄ HOLDING
  6      LITE       ▲   162.9%  ◄ HOLDING
  7      ON         ▲   140.1%  ◄ HOLDING
  8      AMD        ▲   137.3%  ◄ HOLDING
  9      COHR       ▲   120.1%  ◄ HOLDING
  10     GLW        ▲   115.9%  ◄ HOLDING
  11     TER        ▲   106.0%  
  12     LRCX       ▲   104.5%  
  13     HPE        ▲    99.3%  
  14     FIX        ▲    87.3%  
  15     TXN        ▲    83.8%  
  16     GNRC       ▲    83.3%  
  17     AMAT       ▲    78.9%  
  18     MCHP       ▲    78.6%  
  19     SATS       ▲    76.3%  
  20     JBL        ▲    73.1%  

  ... 447 more stocks not shown ...




  MOMENTUM SUMMARY
  ─────────────────────────────────────────────
  Stocks with positive momentum: 280 (60%)
  Avg 6M return (all stocks):   +10.5%
  Avg 6M return (your picks):   +179.9%
  Weakest holding:              GLW (+115.9%)
  Strongest holding:            MU (+310.9%)

  NEXT REBALANCE — check if any of these drop out of top 10:
    Rank #11: TER (+106.0%) — watching
    Rank #12: LRCX (+104.5%) — watching
    Rank #13: HPE (+99.3%) — watching
    Rank #14: FIX (+87.3%) — watching
    Rank #15: TXN (+83.8%) — watching


In [9]:
# backtest
monthly_returns = monthly.pct_change()
port_returns    = (weights.shift(1) *
                   monthly_returns).sum(axis=1)

spy     = yf.download('SPY', start='2015-01-01',
                       auto_adjust=True)['Close'].squeeze()
spy_ret = spy.resample('ME').last().pct_change().squeeze()
spy_cum = (1 + spy_ret).cumprod()
cum     = (1 + port_returns).cumprod()

def sharpe(r, rf=0.02):
    e = r - rf/12
    return float((e.mean() / e.std()) * np.sqrt(12))

def max_drawdown(r):
    c = (1 + r).cumprod()
    return float(((c - c.cummax()) / c.cummax()).min())

def cagr(r):
    c = (1 + r).cumprod()
    y = len(r) / 12
    return float(c.iloc[-1] ** (1/y) - 1)

def win_rate(r):
    return float((r > 0).sum() / len(r))

print(f"Strategy CAGR:   {cagr(port_returns):.1%}")
print(f"SPY CAGR:        {cagr(spy_ret):.1%}")
print(f"Sharpe:          {sharpe(port_returns):.2f}")
print(f"Max Drawdown:    {max_drawdown(port_returns):.1%}")
print(f"Win Rate:        {win_rate(port_returns):.1%}")
print(f"Best Month:      {port_returns.max():.1%}")
print(f"Worst Month:     {port_returns.min():.1%}")

[*********************100%***********************]  1 of 1 completed

Strategy CAGR:   43.5%
SPY CAGR:        14.0%
Sharpe:          1.39
Max Drawdown:    -27.1%
Win Rate:        64.5%
Best Month:      33.5%
Worst Month:     -18.6%


In [ ]:
# Walk-Forward Validation Cell

def walk_forward_sharpe(returns, split=0.70,
                        freq='monthly'):
    split_idx  = int(len(returns) * split)
    in_sample  = returns[:split_idx]
    out_sample = returns[split_idx:]

    # Use 12 for monthly, 252 for daily
    ann = 12 if freq == 'monthly' else 252

    is_sharpe  = float(
        (in_sample.mean() /
         in_sample.std()) * np.sqrt(ann)
    )
    oos_sharpe = float(
        (out_sample.mean() /
         out_sample.std()) * np.sqrt(ann)
    )
    degradation = ((is_sharpe - oos_sharpe) /
                    abs(is_sharpe)) * 100

    print("=== WALK-FORWARD VALIDATION ===")
    print(f"Training period:      "
          f"{returns.index[0].date()} to "
          f"{returns.index[split_idx].date()}")
    print(f"Test period:          "
          f"{returns.index[split_idx].date()} to "
          f"{returns.index[-1].date()}")
    print(f"In-sample Sharpe:     {is_sharpe:.2f}")
    print(f"Out-of-sample Sharpe: {oos_sharpe:.2f}")
    print(f"Degradation:          {degradation:.1f}%")

    if oos_sharpe > is_sharpe:
        print("✓ Strategy improves out of sample")
    elif degradation < 30:
        print("✓ Acceptable degradation — "
              "strategy is robust")
    else:
        print("⚠ High degradation — "
              "possible overfitting")

walk_forward_sharpe(port_returns, freq='monthly')

In [10]:
# charts
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        f'Top 10 Momentum Stocks vs SPY '
        f'(universe: {data.shape[1]} stocks)',
        'Drawdown',
        'Monthly Returns'
    ),
    row_heights=[0.45, 0.25, 0.30],
    vertical_spacing=0.08
)

fig.add_trace(go.Scatter(
    x=cum.index, y=(cum-1)*100,
    name='Momentum',
    line=dict(color='#1D9E75', width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=spy_cum.index, y=(spy_cum-1)*100,
    name='SPY',
    line=dict(color='#888780', width=1.5, dash='dash')
), row=1, col=1)

dd = (cum / cum.cummax() - 1) * 100
fig.add_trace(go.Scatter(
    x=dd.index, y=dd,
    fill='tozeroy',
    line=dict(color='#D85A30', width=1),
    fillcolor='rgba(216,90,48,0.15)',
    name='Drawdown'
), row=2, col=1)

colors = ['#1D9E75' if r >= 0 else '#D85A30'
          for r in port_returns]
fig.add_trace(go.Bar(
    x=port_returns.index,
    y=port_returns * 100,
    marker_color=colors,
    name='Monthly Return'
), row=3, col=1)

fig.update_layout(
    height=800,
    title='Momentum Rotation — S&P 500 Universe',
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig.update_yaxes(ticksuffix='%')
fig.show()

# Momentum ranking — top 20 and bottom 20
top20    = momentum_rets.iloc[-1].nlargest(20)
bottom20 = momentum_rets.iloc[-1].nsmallest(20)
combined = pd.concat([bottom20, top20]).sort_values()

fig2 = go.Figure(go.Bar(
    x=combined.values * 100,
    y=combined.index,
    orientation='h',
    marker_color=['#1D9E75' if t in picks.index
                  else '#D85A30' if v < 0
                  else '#888780'
                  for t, v in combined.items()]
))
fig2.update_layout(
    title='Top 20 & Bottom 20 Momentum Stocks '
          '— green = selected',
    xaxis_title='6-Month Return %',
    height=700,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.update_xaxes(ticksuffix='%')
fig2.show()

In [11]:
# smart rebalance
positions     = client.get_all_positions()
port_value    = float(client.get_account().cash) + \
                sum([float(p.market_value) for p in positions])
current_holds = {p.symbol: float(p.market_value)
                 for p in positions}
target        = {t: port_value * 0.95 * w
                 for t, w in picks.items()}

print(f"Portfolio value: ${port_value:,.2f}")
print(f"Deploying across {len(picks)} stocks\n")

all_tickers = set(current_holds.keys()) | set(target.keys())

print("=== CURRENT vs TARGET ===")
for t in sorted(all_tickers):
    cur  = current_holds.get(t, 0)
    want = target.get(t, 0)
    diff = want - cur
    print(f"  {t}: have ${cur:,.0f} → "
          f"want ${want:,.0f} "
          f"({'buy' if diff>0 else 'SELL'} "
          f"${abs(diff):,.0f})")

print("\n=== PLACING ORDERS ===")

for t in all_tickers:
    cur  = current_holds.get(t, 0)
    want = target.get(t, 0)
    diff = want - cur
    if want == 0 and cur > 0:
        client.close_position(t)
        print(f"  SOLD {t} — out of top 10")
    elif diff < -500:
        client.submit_order(MarketOrderRequest(
            symbol=t,
            notional=round(abs(diff), 2),
            side=OrderSide.SELL,
            time_in_force=TimeInForce.DAY
        ))
        print(f"  Trimmed ${abs(diff):,.0f} of {t}")

for t in all_tickers:
    cur  = current_holds.get(t, 0)
    want = target.get(t, 0)
    diff = want - cur
    if diff > 500:
        client.submit_order(MarketOrderRequest(
            symbol=t,
            notional=round(diff, 2),
            side=OrderSide.BUY,
            time_in_force=TimeInForce.DAY
        ))
        print(f"  Bought ${diff:,.0f} of {t}")
    elif abs(diff) <= 500:
        print(f"  {t}: skipping (too small)")

Portfolio value: $163,567.76
Deploying across 10 stocks

=== CURRENT vs TARGET ===
  AMD: have $15,438 → want $15,539 (buy $101)
  CIEN: have $14,637 → want $15,539 (buy $902)
  COHR: have $15,301 → want $15,539 (buy $238)
  GLW: have $16,587 → want $15,539 (SELL $1,049)
  INTC: have $16,603 → want $15,539 (SELL $1,064)
  LITE: have $14,579 → want $15,539 (buy $960)
  MU: have $16,347 → want $15,539 (SELL $808)
  ON: have $15,757 → want $15,539 (SELL $218)
  STX: have $14,845 → want $15,539 (buy $694)
  WDC: have $15,607 → want $15,539 (SELL $68)

=== PLACING ORDERS ===
  Trimmed $1,049 of GLW
  Trimmed $808 of MU
  Trimmed $1,064 of INTC
  AMD: skipping (too small)
  ON: skipping (too small)
  COHR: skipping (too small)
  Bought $694 of STX
  Bought $960 of LITE
  WDC: skipping (too small)
  Bought $902 of CIEN


In [12]:
# Daily Summary
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("=" * 55)
print(f"  PORTFOLIO SUMMARY")
print(f"  {datetime.now().strftime('%B %d, %Y %I:%M %p')}")
print("=" * 55)

# Account Overview
account    = client.get_account()
cash       = float(account.cash)
equity     = float(account.equity)
last_eq    = float(account.last_equity)
day_pl     = equity - last_eq
day_pl_pct = (day_pl / last_eq) * 100

print(f"\n  ACCOUNT")
print(f"  {'Total equity':<25} ${equity:>12,.2f}")
print(f"  {'Cash':<25} ${cash:>12,.2f}")
print(f"  {'Invested':<25} ${equity-cash:>12,.2f}")
print(f"  {'Today P&L':<25} "
      f"{'▲' if day_pl >= 0 else '▼'}"
      f" ${abs(day_pl):>10,.2f} "
      f"({day_pl_pct:+.2f}%)")

# Positions
positions = client.get_all_positions()

if not positions:
    print("\n  No open positions — fully in cash")
else:
    print(f"\n  POSITIONS ({len(positions)} stocks)\n")
    print(f"  {'Ticker':<8} {'Shares':>7} {'Entry':>9} "
          f"{'Current':>9} {'Value':>10} "
          f"{'Day %':>8} {'Total %':>8}")
    print(f"  {'-'*8} {'-'*7} {'-'*9} "
          f"{'-'*9} {'-'*10} "
          f"{'-'*8} {'-'*8}")

    total_unrealized = 0
    winners          = []
    losers           = []

    for p in sorted(positions,
                    key=lambda x: float(x.unrealized_plpc),
                    reverse=True):
        symbol        = p.symbol
        qty           = float(p.qty)
        entry         = float(p.avg_entry_price)
        current       = float(p.current_price)
        mkt_val       = float(p.market_value)
        unrealized    = float(p.unrealized_pl)
        unrealized_pc = float(p.unrealized_plpc) * 100
        day_pc        = float(p.change_today) * 100
        total_unrealized += unrealized

        # Arrow indicators
        day_arrow   = '▲' if day_pc >= 0 else '▼'
        total_arrow = '▲' if unrealized_pc >= 0 else '▼'

        print(f"  {symbol:<8} "
              f"{qty:>7.1f} "
              f"${entry:>8.2f} "
              f"${current:>8.2f} "
              f"${mkt_val:>9,.0f} "
              f"{day_arrow}{abs(day_pc):>6.2f}% "
              f"{total_arrow}{abs(unrealized_pc):>6.2f}%")

        if unrealized_pc > 0:
            winners.append((symbol, unrealized_pc))
        else:
            losers.append((symbol, unrealized_pc))

    # P&L Summary
    print(f"\n  {'─'*53}")
    unreal_arrow = '▲' if total_unrealized >= 0 else '▼'
    print(f"  {'Total unrealized P&L':<25} "
          f"{unreal_arrow} ${abs(total_unrealized):>9,.2f}")

    # Best and worst performers
    if winners:
        best = max(winners, key=lambda x: x[1])
        print(f"  {'Best position':<25} "
              f"{best[0]} (+{best[1]:.2f}%)")
    if losers:
        worst = min(losers, key=lambda x: x[1])
        print(f"  {'Worst position':<25} "
              f"{worst[0]} ({worst[1]:.2f}%)")

    # Portfolio health
    print(f"\n  PORTFOLIO HEALTH")
    n_up   = len(winners)
    n_down = len(losers)
    total  = len(positions)
    bar    = ('▲' * n_up) + ('▼' * n_down)
    print(f"  {n_up} up / {n_down} down  {bar}")

    avg_return = np.mean(
        [float(p.unrealized_plpc) * 100
         for p in positions]
    )
    print(f"  Avg position return:      {avg_return:+.2f}%")

    # Stop loss warnings
    warnings = [(p.symbol, float(p.unrealized_plpc)*100)
                for p in positions
                if float(p.unrealized_plpc) < -0.10]

    if warnings:
        print(f"\n  ⚠ STOP LOSS WARNINGS (down >10%)")
        for symbol, pct in warnings:
            print(f"    {symbol}: {pct:.2f}% — "
                  f"consider reviewing")

# Recent orders
print(f"\n  RECENT ORDERS (last 5)")
print(f"  {'─'*53}")

try:
    orders = client.get_orders(
        filter=dict(limit=5, status='all')
    )
    if not orders:
        print("  No recent orders")
    else:
        for o in orders:
            side   = o.side.value.upper()
            symbol = o.symbol
            status = o.status.value
            filled = o.filled_at
            amt    = (f"${float(o.notional):,.0f}"
                      if o.notional
                      else f"{float(o.qty)} shares")
            date   = (filled.strftime('%b %d %I:%M%p')
                      if filled else 'pending')
            print(f"  {side:<5} {symbol:<8} {amt:>10} "
                  f"  {status:<10} {date}")
except Exception as e:
    print(f"  Could not load orders: {e}")

# Quick stats on holdings
if positions:
    print(f"\n  HOLDING STATS")
    print(f"  {'─'*53}")

    tickers_held = [p.symbol for p in positions]
    hist = yf.download(
        tickers_held,
        period='1mo',
        auto_adjust=True,
        progress=False
    )['Close']

    if isinstance(hist, pd.Series):
        hist = hist.to_frame()

    hist   = hist.squeeze(axis=None)
    rets   = hist.pct_change(fill_method=None).dropna()

    for ticker in tickers_held:
        if ticker not in rets.columns:
            continue
        r      = rets[ticker].dropna()
        vol    = r.std() * np.sqrt(252) * 100
        mo_ret = ((1 + r).prod() - 1) * 100
        print(f"  {ticker:<8} "
              f"1mo return: {mo_ret:>+7.2f}%  "
              f"annualized vol: {vol:>5.1f}%")

print(f"\n{'=' * 55}\n")

  PORTFOLIO SUMMARY
  June 18, 2026 10:23 PM

  ACCOUNT
  Total equity              $  163,564.07
  Cash                      $    7,867.40
  Invested                  $  155,696.67
  Today P&L                 ▲ $  7,553.81 (+4.84%)

  POSITIONS (10 stocks)

  Ticker    Shares     Entry   Current      Value    Day %  Total %
  -------- ------- --------- --------- ---------- -------- --------
  MU          14.2 $  417.84 $ 1150.74 $   16,343 ▲ 10.31% ▲175.40%
  WDC         20.6 $  347.32 $  756.33 $   15,607 ▲  6.21% ▲117.76%
  COHR        39.1 $  311.46 $  391.15 $   15,301 ▲  3.25% ▲ 25.59%
  INTC       124.0 $  121.12 $  133.94 $   16,603 ▲ 10.60% ▲ 10.58%
  GLW         85.3 $  176.07 $  194.52 $   16,587 ▲ 10.90% ▲ 10.48%
  ON         129.1 $  116.31 $  122.06 $   15,757 ▲  8.10% ▲  4.95%
  AMD         28.7 $  522.44 $  537.20 $   15,438 ▲  4.82% ▲  2.83%
  STX         13.8 $ 1085.99 $ 1073.75 $   14,845 ▲  0.72% ▼  1.13%
  LITE        17.2 $  897.63 $  850.00 $   14,579 ▼  2.30% ▼ 

In [17]:
# Fixed metrics for MONTHLY returns
def sharpe_monthly(r, rf=0.02):
    e = r - rf/12  # monthly rf
    return float((e.mean() / e.std()) * np.sqrt(12))

def sortino_monthly(r, rf=0.02):
    e        = r - rf/12
    neg      = e[e < 0]
    downside = np.sqrt((neg**2).mean()) * np.sqrt(12)
    return float(e.mean() * 12 / downside)

def max_drawdown(r):
    c = (1 + r).cumprod()
    return float(((c - c.cummax()) / c.cummax()).min())

def cagr_monthly(r):
    c = (1 + r).cumprod()
    y = len(r) / 12  # months not days
    return float(c.iloc[-1] ** (1/y) - 1)

def win_rate(r):
    return float((r > 0).sum() / len(r))

# SPY monthly
spy        = yf.download('SPY', start='2015-01-01',
                          auto_adjust=True,
                          progress=False)['Close'].squeeze()
spy_monthly = spy.resample('ME').last().pct_change(
    fill_method=None
).squeeze()

# Align dates
common      = port_returns.index.intersection(
    spy_monthly.index
)
port_aligned = port_returns[common]
spy_aligned  = spy_monthly[common]

print("=" * 40)
print("MOMENTUM — FIXED BACKTEST RESULTS")
print("=" * 40)
print(f"Strategy CAGR:    {cagr_monthly(port_aligned):.1%}")
print(f"SPY CAGR:         {cagr_monthly(spy_aligned):.1%}")
print(f"Sharpe Ratio:     {sharpe_monthly(port_aligned):.2f}")
print(f"Sortino Ratio:    {sortino_monthly(port_aligned):.2f}")
print(f"Max Drawdown:     {max_drawdown(port_aligned):.1%}")
print(f"Win Rate:         {win_rate(port_aligned):.1%}")
print(f"SPY Correlation:  "
      f"{port_aligned.corr(spy_aligned):.2f}")

MOMENTUM — FIXED BACKTEST RESULTS
Strategy CAGR:    43.5%
SPY CAGR:         14.0%
Sharpe Ratio:     1.39
Sortino Ratio:    1.72
Max Drawdown:     -27.1%
Win Rate:         64.5%
SPY Correlation:  0.70


In [14]:
# Momentum diagnostic
print("Weights sum check (should be ~1.0):")
print(weights.sum(axis=1).tail(10))

print("\nPort returns sample (should be small %):")
print(port_returns.tail(20))

print("\nAny extreme values?")
print(port_returns.describe())

print("\nMax single day return:")
print(port_returns.max())

print("\nMin single day return:")
print(port_returns.min())

Weights sum check (should be ~1.0):
Date
2025-09-30    1.0
2025-10-31    1.0
2025-11-30    1.0
2025-12-31    1.0
2026-01-31    1.0
2026-02-28    1.0
2026-03-31    1.0
2026-04-30    1.0
2026-05-31    1.0
2026-06-30    1.0
Freq: ME, dtype: float64

Port returns sample (should be small %):
Date
2024-11-30    0.095505
2024-12-31   -0.084682
2025-01-31    0.064423
2025-02-28   -0.035805
2025-03-31   -0.155524
2025-04-30   -0.010739
2025-05-31    0.008656
2025-06-30    0.072690
2025-07-31    0.005687
2025-08-31   -0.006407
2025-09-30    0.170808
2025-10-31    0.155315
2025-11-30    0.077559
2025-12-31    0.122767
2026-01-31    0.212486
2026-02-28    0.192456
2026-03-31   -0.030547
2026-04-30    0.334646
2026-05-31    0.119525
2026-06-30    0.076220
Freq: ME, dtype: float64

Any extreme values?
count    138.000000
mean       0.033595
std        0.079747
min       -0.185962
25%       -0.012494
50%        0.030327
75%        0.076214
max        0.334646
dtype: float64

Max single day return:
0.

In [16]:
# Kelly Criterion
def kelly_fraction(returns):
    wr    = float((returns > 0).mean())
    avg_w = float(returns[returns > 0].mean())
    avg_l = float(abs(returns[returns < 0].mean()))
    f     = (wr / avg_l) - ((1 - wr) / avg_w)
    return max(0, min(f / 2, 0.25))

kelly = kelly_fraction(port_returns)
print(f"Kelly fraction: {kelly:.1%}")

Kelly fraction: 25.0%
